# Case Biossensor - Organização, validação e modelagem

Este notebook executa o pipeline completo solicitado no case.

## 1) Configuração
Defina os caminhos de entrada e saída antes de executar.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.append(str(ROOT / 'src'))

from biosensor_pipeline import (
    run_pipeline,
    validate_voltammograms,
    build_structured_tables
)

INPUT_DIR = ROOT / 'data' / 'raw'  # ajuste para o diretório real
OUTPUT_DIR = ROOT / 'outputs'
INPUT_DIR, OUTPUT_DIR

## 2) Execução do pipeline
Leitura, validação, organização do banco e exportação.

In [ ]:
result = run_pipeline(INPUT_DIR, OUTPUT_DIR, random_state=42)
result['long'].head()

## 3) Qualidade dos dados e estrutura
Verificações de duplicidade, dados ausentes, formato e consistência.

In [ ]:
raw = result['raw']
validation = validate_voltammograms(raw)
validation

In [ ]:
long_df, wide_df, metadata_df = build_structured_tables(raw)
long_df.shape, wide_df.shape, metadata_df.shape

## 4) Visualização inicial dos voltamogramas
Comparação entre condições do sensor.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9,5))
for condition, group in long_df.groupby('sensor_condition'):
    curve = group.groupby('potential', as_index=False)['current'].mean()
    ax.plot(curve['potential'], curve['current'], label=condition)
ax.set_xlabel('Potencial')
ax.set_ylabel('Corrente')
ax.set_title('Voltamogramas médios por condição')
ax.legend()
plt.show()

## 5) Modelagem e métricas
Modelos baseline e ML com separação por amostra para evitar vazamento.

In [ ]:
result['model_metrics']

In [ ]:
result['predictions'].head() if result['predictions'] is not None else 'Dados insuficientes para gerar previsões'

In [ ]:
result['feature_importance'].head(10) if result['feature_importance'] is not None else 'Dados insuficientes para importância de features'

## 6) Artefatos gerados
Arquivos exportados em CSV e banco SQLite no diretório de saída.

In [ ]:
sorted([p.name for p in OUTPUT_DIR.glob('*')])